# Day 019：MoE 参数量、激活参数量与计算成本

本 Notebook 沿 Day 018 的路由流程继续，使用真实 MiniMind 模块验证普通 FFN 与 MoE 的总参数量、每个 token 的激活路径、expert-token routes 和近似计算量。

In [ ]:
import sys
from pathlib import Path
candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(root for root in candidate_roots if (root / 'minimind' / 'model' / 'model_minimind.py').exists())
sys.path.insert(0, str(repo_root / 'minimind'))
import torch
from model.model_minimind import MiniMindConfig, FeedForward, MOEFeedForward, MiniMindForCausalLM
torch.manual_seed(0)
print('repo:', repo_root)

## 1. 一套普通 FFN 的参数量

三张无 bias 矩阵各有 `H*I` 个参数，因此普通 FFN 的参数量是 `3*H*I`。

In [ ]:
H, I = 16, 32
config = MiniMindConfig(hidden_size=H, intermediate_size=I)
ffn = FeedForward(config)
for name, parameter in ffn.named_parameters():
    print(name, tuple(parameter.shape), parameter.numel())
print('formula:', 3 * H * I)
print('total:', sum(parameter.numel() for parameter in ffn.parameters()))

## 2. MoE MLP 的总参数量

MoE 保存 `E` 个完整 expert，再保存一个 `H -> E` 的 router。总参数量为 `E*(3*H*I) + E*H`。

In [ ]:
E = 4
config = MiniMindConfig(hidden_size=H, num_experts=E, moe_intermediate_size=I)
moe = MOEFeedForward(config)
expert_params = sum(p.numel() for p in moe.experts[0].parameters())
router_params = moe.gate.weight.numel()
print('one expert:', expert_params)
print('router:', router_params)
print('formula:', E * expert_params + router_params)
print('total:', sum(p.numel() for p in moe.parameters()))

## 3. 完整模型：普通 FFN 与 MoE

只切换 `use_moe`，Embedding、Attention、归一化和 tied 的输出头保持相同；参数差值来自每层 MLP 的替换。

In [ ]:
common = dict(
    vocab_size=20, hidden_size=8, num_hidden_layers=2,
    num_attention_heads=2, num_key_value_heads=1,
    intermediate_size=16, moe_intermediate_size=16,
    num_experts=4, num_experts_per_tok=1,
    max_position_embeddings=32, flash_attn=False
)
totals = {}
for use_moe in (False, True):
    model = MiniMindForCausalLM(MiniMindConfig(use_moe=use_moe, **common))
    totals[use_moe] = sum(p.numel() for p in model.parameters())
    print('use_moe =', use_moe, '| total params =', totals[use_moe])
print('difference:', totals[True] - totals[False])

## 4. 激活参数量：`k=1` 与 `k=2`

总参数量包含所有 expert；单个 token 只经过 router 和 `k` 个 expert。近似激活参数量是 `E*H + k*(3*H*I)`。

In [ ]:
H, I, E = 8, 16, 4
ordinary = 3 * H * I
router = E * H
print('ordinary FFN active params:', ordinary)
for k in (1, 2):
    print(f'MoE k={k} active params:', router + k * ordinary)

## 5. expert-token routes

输入有 `T` 个 token、每个 token 选 `k` 个 expert 时，route 数为 `T*k`。hook 看到的是每个 expert 收到的 token 行数；这些行数的总和就是 route 数。

In [ ]:
def inspect_routes(k):
    config = MiniMindConfig(hidden_size=8, num_experts=4, num_experts_per_tok=k, moe_intermediate_size=16)
    moe = MOEFeedForward(config).eval()
    calls = []
    handles = [expert.register_forward_hook(lambda module, inputs, output: calls.append(inputs[0].shape[0])) for expert in moe.experts]
    _ = moe(torch.randn(2, 3, 8))
    for handle in handles:
        handle.remove()
    print(f'k={k}: per-expert rows={calls}, total routes={sum(calls)}, expected={2*3*k}')

inspect_routes(1)
inspect_routes(2)

## 6. 近似计算成本

忽略 dispatch 和激活函数，一次矩阵乘加按一个 MAC 近似：普通 FFN 是 `3*H*I`，router 是 `H*E`，MoE experts 是 `k*3*H*I`。实际大型模型中，主要成本随 `k` 增长。

In [ ]:
ordinary_macs = 3 * H * I
print('ordinary FFN MACs/token:', ordinary_macs)
for k in (1, 2):
    moe_macs = E * H + k * ordinary_macs
    print(f'MoE k={k} MACs/token:', moe_macs, '| ratio:', moe_macs / ordinary_macs)

## 今日结论与 Day 020 恢复点

MoE 通过保存多套 expert 增加总容量；通过 `num_experts_per_tok` 控制每个 token 实际激活的专家路径。所有 expert 权重仍需存储，稀疏激活主要减少计算，不会自动减少权重显存。

Day 020：进入 `train_pretrain.py` / `train_full_sft.py` 的 `--use_moe` 分支，完成一次极小规模 MoE forward、backward 和 optimizer.step，检查 router 与 expert 的梯度。